# Day 2 (teammate A) - ASL words. CPU is enough; no Day-1 data needed.
Inputs: `slr-code` and the competition **Google - Isolated Sign Language Recognition** (https://www.kaggle.com/competitions/asl-signs: click *Join Competition* / accept the rules once, then *+ Add Input -> Competition Data*). Internet ON. Never download the data; it is only attached.

In [ ]:
# Finds the uploaded "slr-code" dataset wherever Kaggle mounts it (the path differs between Kaggle versions).
import glob, os, shutil, subprocess, sys, zipfile
ON_KAGGLE = os.path.exists("/kaggle/input")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
CODE = f"{WORK}/code"
hits = [h for d in ("*", "*/*", "*/*/*") for h in glob.glob(f"/kaggle/input/{d}/slr/__init__.py")] + glob.glob("/content/slr_code/**/slr/__init__.py", recursive=True)  # shallow search: ASL Alphabet has ~87k files
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE, dirs_exist_ok=True)
else:
    zips = [h for d in ("*", "*/*", "*/*/*") for h in glob.glob(f"/kaggle/input/{d}/slr_code.zip")] + glob.glob("/content/**/slr_code.zip", recursive=True)
    if zips:
        zipfile.ZipFile(zips[0]).extractall(CODE)
    else:
        print("slr-code NOT FOUND. Add it: right sidebar -> + Add Input -> Your Datasets -> slr-code. Currently attached:")
        for p in sorted(glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*")):
            print("  ", p)
        raise SystemExit("Attach the slr-code dataset, then run this cell again.")
os.chdir(CODE)
print("code ready in", CODE, os.listdir(CODE))
# Install each package on its own so one failure cannot skip the others. Kaggle now runs Python 3.13, so we use the
# current mediapipe (Tasks API) - not the old pinned version.
for pkg in ("mediapipe", "remotezip", "opencv-python-headless"):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    print(pkg, "OK" if r.returncode == 0 else "FAILED\n" + r.stderr[-600:])
import mediapipe; print("mediapipe", mediapipe.__version__)

In [ ]:
import glob, os
def find_dir(marker, root="/kaggle/input", depth=5):
    """Folder that directly contains `marker` (file or folder), searched shallowly - Kaggle mount paths differ between versions."""
    for d in range(depth + 1):
        for p in glob.glob(root + "/*" * d + "/" + marker):
            return os.path.dirname(p)
    return None

## 1. Find the data and pick the signs

In [ ]:
ASL_DATA = find_dir("train_landmark_files")
assert ASL_DATA, "asl-signs competition data not attached: + Add Input -> Competition Data -> asl-signs (accept the rules first)"
print("data:", ASL_DATA)
SIGNS = "hello thankyou please yes no mom dad water eat drink go sleep happy sad like want home cat dog bird red blue green hungry sick".split()

## 2. Convert only these signs to keypoints and train (about 20-40 minutes on CPU)

In [ ]:
!python scripts/asl_words.py extract --data $ASL_DATA --signs {" ".join(SIGNS)} --per-sign 300 --out $WORK/asl_words
!python scripts/train_words.py --data $WORK/asl_words --out $WORK/models/asl_words.pt --epochs 60

## 3. (Optional) pretrained model benchmark - skip unless the team lead asks
Needs a pretrained TFLite ASL model attached as an input; set `RUN_PRETRAINED = True` and `MODEL_DIR`.

In [ ]:
RUN_PRETRAINED = False
MODEL_DIR = "/kaggle/input/sign-language"
if RUN_PRETRAINED:
    !python scripts/asl_words.py infer --model-dir $MODEL_DIR --data $ASL_DATA --signs {" ".join(SIGNS)} --per-sign 20 --out $WORK/models/asl_pretrained_eval.json

## 4. Package for download

In [ ]:
!cd $WORK && zip -r asl_models.zip models && ls -lh asl_models.zip

Then **Save Version -> Save & Run All**, open the Output tab and download `asl_models.zip`.